In [ ]:
# import libraries
import os
from pathlib import Path
import pandas as pd
import ee
import geemap

import numpy as np

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    f1_score,
)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Authenticate GEE
ee.Authenticate()
ee.Initialize()
#print("Google Earth Engine initialized successfully!")

Parameters

In [ ]:
GLC_URBAN_CODE = 190          # "Impervious_surfaces" class in GLC_FCS30D legend
START_YEAR = 2015
END_YEAR = 2022   

Path

In [ ]:
# Repo/folder directory
PROJECT_DIR = Path.cwd().parent

# Output paths
OUTPUT_DIR = PROJECT_DIR / "Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True) 
print(OUTPUT_DIR.resolve())

# For thematic layers
OUT_THEMATIC  = OUTPUT_DIR / "Thematic"
OUT_THEMATIC.mkdir(parents=True, exist_ok=True)

# For urban layers
OUT_URBAN  = OUTPUT_DIR / "Urban"
OUT_URBAN.mkdir(parents=True, exist_ok=True)

In [ ]:
# Center coordinates to show map
fct_center =  (9.056266, 7.498522)

## Visualise Parameters

In [ ]:
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters 
vis_params_s2_rgb = {"min" : 0, "max" :0.3, "bands": ["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min" : 0, "max" :0.3, "bands": ["B8", "B4", "B3"]}


# Spectral indices visualization parameters
# NDVI
vis_params_ndvi = {
    "min": -0.2,
    "max": 0.8,
    "palette": [
        "#a50026",  
        "#d73027",
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#66bd63",
        "#1a9850",
        "#006837",  
    ],
}

# NDBI
vis_params_ndbi = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#f7f7f7",  
        "#fddbc7",
        "#f4a582",
        "#d6604d",
        "#b2182b",  
    ],
}

# NDWI
vis_params_ndwi = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#f7fbff",  
        "#deebf7",
        "#9ecae1",
        "#4292c6",
        "#2171b5",
        "#08306b",  
    ],
}


# Visualisation parameters for road layers
vis_params_roads_vector = {
                        "color": "red",
                        "width": 1.5,
                    }

vis_params_roads_raster = {
                        "min": 0,
                        "max": 1,
                        "palette": ["black", "white"],
                    }


vis_params_dist_road = {
    "min": 0,
    "max": 21730, #meters
    "palette": [
        "#d73027",  
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#1a9850",  
    ],
}

# Visualisation parameters for water layers
vis_params_water = {
    "min": 0,
    "max": 1,
    "palette": ["white", "blue"], 
}

# Distance to water
vis_params_dist_water = {
    "min": 0,
    "max": 38497,  
    "palette": [
        "#f7fbff",  
        "#deebf7",
        "#9ecae1",
        "#4292c6",
        "#2171b5",
        "#08306b",  
    ],
}


# Nighttime lights visualisation parameters
vis_params_ntl = {
    "min": 0,
    "max": 20,  
    "palette": [
        "#000000",  
        "#2c0b00",
        "#6e1c00",
        "#a83800",
        "#d9720a",
        "#f7b733",
        "#ffe98a",  
    ],
}


# Visualization parameters for GPWv411 population DENSITY layer
vis_params_gpw = {
  "min": 0.0,
  "max": 10000.0,
  "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
}

suitability_vis = {"min": 0, "max": 1, "palette": ["ffffcc", "fd8d3c", "800026"]}

Base Map

In [ ]:
# Map to visualize the FAO GAUL boundary data
boundary_map = geemap.Map(center=(7.0, 8.0), zoom=10)

# Map to visualise land cover layers
lc_map = geemap.Map(center = fct_center, zoom=8)
lc_map.add_basemap("SATELLITE")

# Map to visualize Sentinel-2
s2_map = geemap.Map(center=fct_center, zoom=8)


# Maps to visualise thematic layers
thematic_map_1 = geemap.Map(center = fct_center, zoom=8)
thematic_map_1.add_basemap("SATELLITE")

thematic_map_2 = geemap.Map(center = fct_center, zoom=8)
thematic_map_2.add_basemap("SATELLITE")


# Map to visualise ML results
ml_map = geemap.Map(center = fct_center, zoom=8)
ml_map.add_basemap("SATELLITE")

suitability_map = geemap.Map(center=fct_center, zoom=10)
suitability_map.add_basemap("SATELLITE")


Helper Functions

In [ ]:
# Function to export Image to Drive
def export_image_to_drive(image: ee.Image, description: str, folder: str, extent: ee.Geometry, scale: int, crs: str = 'EPSG:4326'):
    """
    Starts a Google Earth Engine batch task to export a raster image to Google Drive.

    Args:
        image (ee.Image): The Earth Engine image to be exported.
        description (str): A human-readable name for the task and the output file prefix.
        folder (str): The name of the Google Drive folder where the file will be saved.
        extent (ee.Geometry): The regional boundary (AOI) to clip the export to.
        scale (int): The resolution in meters per pixel (e.g., 10 for Sentinel-2).
        crs (str): The coordinate reference system.

    Returns:
        None: The function initiates an asynchronous background task.
    """
    # Convert all bands to Float32 (or Float64)
    image = image.toFloat()

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        region=extent,
        scale=scale,
        crs=crs,
        maxPixels=1e13
    )
    task.start()
    return task
    print(f"Started export: {description}")

In [ ]:
# Function to export Image to Drive
def export_image_to_drive(image: ee.Image, description: str, folder: str, extent: ee.Geometry, scale: int, crs: str = 'EPSG:4326'):
    """
    Starts a Google Earth Engine batch task to export a raster image to Google Drive.

    Args:
        image (ee.Image): The Earth Engine image to be exported.
        description (str): A human-readable name for the task and the output file prefix.
        folder (str): The name of the Google Drive folder where the file will be saved.
        extent (ee.Geometry): The regional boundary (AOI) to clip the export to.
        scale (int): The resolution in meters per pixel (e.g., 10 for Sentinel-2).
        crs (str): The coordinate reference system.

    Returns:
        None: The function initiates an asynchronous background task.
    """
    # Convert all bands to Float32 (or Float64)
    image = image.toFloat()

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        region=extent,
        scale=scale,
        crs=crs,
        maxPixels=1e13
    )
    task.start()
    return task
    print(f"Started export: {description}")

Boundary Data

In [ ]:
# Boundary data from FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0
fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Countries boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # States boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2') # LGAs boundaries

styleParams = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

#dataset = dataset.style(styleParams)


boundary_map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
boundary_map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
boundary_map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
boundary_map

In [ ]:
# Select just a single feature
print(fao_gaul_l0.limit(1).getInfo()["columns"])
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja")) # FCT extent
print(fct_l0.getInfo())

# Get geometry from FCT boundary (FeatureCollection)
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

# Creat map to visualize FCT boundary data
aoi_map = geemap.Map(center=fct_center, zoom=10)
aoi_map.addLayer(fct_l0, vis_params_aoi, 'FCT Boundary')
aoi_map

In [ ]:
# Image Collection
# Data Source: 
s2_img_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') # All Sentinel-2 collection
    .filterDate('2022-01-01', '2022-01-31') # Limit/filter to January 2022
    .filterBounds(fct_l0.geometry()) # Limit/filter to Abuja
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# Apply scaling factor
def apply_scale(image):
    scaled_image = image.multiply(0.0001).copyProperties(image, ['system:time_start'])
    return  scaled_image

s2_img_col = s2_img_col.map(apply_scale)

# Collection Properties
#print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}\n")


# First image in collection
first_s2_img = s2_img_col.first()
#print(f"First image in S2 collection: {first_s2_img.getInfo()}\n")

# Bands in S2 image
#print(first_s2_img.bandNames().getInfo())

# Select just a few bands
red_band_s2 = first_s2_img.select(["B4", "B3", "B2"])
#print(f"Red Band: {red_band_s2.bandNames().getInfo()}")


# Visualize S2 image
s2_map.addLayer(fct_l0, vis_params_aoi, 'FCT Boundary')
s2_map.addLayer(first_s2_img.clip(aoi), vis_params_s2_rgb, "First S2 Img")
s2_map

In [ ]:
# Mosaic the entire S2 collection & clip to FCT extent [Spatial Mosaic]
s2_mosaic = s2_img_col.mosaic()
s2_mosaic_clipped = s2_mosaic.clip(aoi)
#print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

s2_map.addLayer(s2_mosaic_clipped, vis_params_s2_rgb, "S2 Mosaiced - Spatial")
s2_map

In [ ]:
# Compute median composite the entire S2 collection & clip to FCT extent [Temporal Composite]
s2_median = s2_img_col.median().clip(aoi)
print(f"Median of S2 collection: {s2_median.getInfo()}")

# Compute spectral indices
ndvi = s2_median.normalizedDifference(["B8", "B4"]).rename("ndvi")
ndwi = s2_median.normalizedDifference(["B3", "B8"]).rename("ndwi")

s2_map.addLayer(s2_median, vis_params_s2_fcc, "S2 Median Comp")
s2_map.addLayer(ndvi, vis_params_ndvi, "S2 NDVI")
s2_map.addLayer(ndwi, vis_params_ndwi, "S2 NDWI")
s2_map

 Land Cover / Existing Urban Layer (GLC_FCS30D: 2012–2022)
The raw GLC_FCS30D asset is stored as tiled, multi-band images (one band per year). We mosaic the tiles, rename the bands to actual years, and convert the multi-band image into a proper year-indexed ImageCollection.

In the GLC_FCS30D legend, class code 190 = Impervious surfaces (built-up / urban). We use this to derive a binary urban mask for every year.

In [ ]:
ee.List.sequence(2000, 2022)

In [ ]:
# GLC_FCS30D Global Land Cover
# Data source : https://gee-community-catalog.org/projects/glc_fcs/?h=glc+fcs30d
# Reference   : https://gee-community-catalog.org/tutorials/examples/glc_fcs30d_lulc/
glc_annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
glc_mosaic = glc_annual.mosaic()

# Mosaic tiled images and rename bands b1, b2, ... to 2000, 2001, ...
# Each image in annual land cover data has 23 bands, one for each year from 2000-2022 (23 years)
glc_years = ee.List.sequence(2000, 2022).map(lambda y: ee.Number(y).format("%04d"))
glc_mosaic_renamed = glc_mosaic.rename(glc_years)

#print("GLC_FCS30D bands Original:", glc_mosaic.bandNames().slice(0, 5).getInfo(), "...")
#print("GLC_FCS30D bands renamed:", glc_mosaic_renamed.bandNames().slice(0, 5).getInfo(), "...")

In [ ]:
# Classification scheme
# (35 landcover class and 1 fill value)
lc_class_values =  [
  10, 11, 12, 20, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 120, 121, 122, 
  130, 140, 150, 152, 153, 181, 182, 183, 184, 185, 186, 187, 190, 200, 
  201, 202, 210, 220, 0
]

# Land cover class names
glc_class_names = [
    "Rainfed_cropland", "Herbaceous_cover_cropland", "Tree_or_shrub_cover_cropland",
    "Irrigated_cropland", "Open_evergreen_broadleaved_forest", "Closed_evergreen_broadleaved_forest",
    "Open_deciduous_broadleaved_forest", "Closed_deciduous_broadleaved_forest",
    "Open_evergreen_needle_leaved_forest", "Closed_evergreen_needle_leaved_forest",
    "Open_deciduous_needle_leaved_forest", "Closed_deciduous_needle_leaved_forest",
    "Open_mixed_leaf_forest", "Closed_mixed_leaf_forest", "Shrubland",
    "Evergreen_shrubland", "Deciduous_shrubland", "Grassland", "Lichens_and_mosses",
    "Sparse_vegetation", "Sparse_shrubland", "Sparse_herbaceous", "Swamp", "Marsh",
    "Flooded_flat", "Saline", "Mangrove", "Salt_marsh", "Tidal_flat",
    "Impervious_surfaces", "Bare_areas", "Consolidated_bare_areas",
    "Unconsolidated_bare_areas", "Water_body", "Permanent_ice_and_snow", "Filled_value",
]

glc_class_colours = [
    "#ffff64", "#ffff64", "#ffff00", "#aaf0f0", "#4c7300",
    "#006400", "#a8c800", "#00a000", "#005000", "#003c00",
    "#286400", "#285000", "#a0b432", "#788200", "#966400",
    "#964b00", "#966400", "#ffb432", "#ffdcd2", "#ffebaf",
    "#ffd278", "#ffebaf", "#00a884", "#73ffdf", "#9ebb3b",
    "#828282", "#f57ab6", "#66cdab", "#444f89", "#c31400",
    "#fff5d7", "#dcdcdc", "#fff5d7", "#0046c8", "#ffffff", "#ffffff",
]

In [ ]:
# Visualise land cover for FCT/Abuja
lc_map.addLayer(glc_mosaic_renamed.select("2015").clip(aoi), {"palette": glc_class_colours}, "Land cover - 2015")
lc_map.addLayer(glc_mosaic_renamed.select("2022").clip(aoi), {"palette": glc_class_colours}, "Land cover - 2022")
lc_map

In [ ]:
print(START_YEAR)
print(END_YEAR)

In [ ]:
# Extract only Urban class from land cover 
def get_glc_urban_mask(year):
    '''Return a binary urban (1) / non-urban (0) image for a given year
    from the GLC_FCS30D dataset, clipped to the AOI.'''
    band_name = str(year)
    year_image = glc_mosaic_renamed.select([band_name])
    urban_mask = year_image.eq(GLC_URBAN_CODE).rename("urban")
    return urban_mask.clip(aoi).set({"year": year, "source": "GLC_FCS30D"})

# 
urban_by_year = {
    year: get_glc_urban_mask(year) for year in range(START_YEAR, END_YEAR + 1)
}
all_years = sorted(urban_by_year.keys())
#print("Urban time series (GLC_FCS30D only) covers:", all_years)

In [ ]:
urban_vis = {"min": 0, "max": 1, "palette": ["00000000", "d7301f"]}

Map = geemap.Map()
Map.centerObject(aoi, 10)
lc_map.addLayer(fct_l0.style(color="black", fillColor="00000000"), {}, "FCT Boundary")

for year in [2015, 2018, 2022]:
    lc_map.addLayer(urban_by_year[year], urban_vis, f"Urban {year}")
lc_map

Annual Urban Area Change

In [ ]:
def compute_urban_area_km2(urban_image, region=aoi, scale=30):
    '''Return total urban area in km^2 for a binary urban image.'''
    area_image = urban_image.multiply(ee.Image.pixelArea())
    stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=scale,
        maxPixels=1e10,
    )
    area_m2 = ee.Number(stats.get("urban"))
    return area_m2.divide(1e6)  # convert m^2 -> km^2

urban_area_records = []
for year in all_years:
    area_km2 = compute_urban_area_km2(urban_by_year[year], scale=30).getInfo()
    urban_area_records.append({"year": year, "urban_area_km2": area_km2})
    #print(f"{year}: {area_km2:.2f} km^2")

# To data frame
urban_area_df = pd.DataFrame(urban_area_records)
#urban_area_df

In [ ]:
# Annual expansion rate (km^2/year and %/year)
urban_area_df["expansion_km2"] = urban_area_df["urban_area_km2"].diff()
urban_area_df["expansion_pct"] = urban_area_df["urban_area_km2"].pct_change() * 100
#urban_area_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(urban_area_df["year"], urban_area_df["urban_area_km2"], marker="o", color="firebrick")
ax1.set_xlabel("Year")
ax1.set_ylabel("Urban area (km²)", color="firebrick")
ax1.set_title("FCT/Abuja — Urban Extent Trend (2015–2022, GLC_FCS30D)")

ax2 = ax1.twinx()
ax2.bar(urban_area_df["year"], urban_area_df["expansion_pct"], alpha=0.3, color="steelblue")
ax2.set_ylabel("Annual growth rate (%)", color="steelblue")
plt.tight_layout()
plt.savefig(OUT_URBAN / "urban_growth_trend.png", dpi=200)
plt.show()

Drivers of Urban Growth / Predictors (Thematic Layers)
Note that the driver of urban growth has different temporal resolution. Where possible, we used date close to 2022. 2022 is the later year in the land cover/urban change data.

Elevation model

In [ ]:
# Download elevation and compute slope
# Data source: https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003
dem = ee.Image('USGS/SRTMGL1_003')
#print(f"Bands in the DEM {dem.bandNames().getInfo()}")


elevation = dem.select('elevation')
slope = ee.Terrain.slope(elevation)

thematic_map_1.addLayer(dem.clip(aoi), {'min': 0, 'max': 500, "palette": ["Red", "Green", "Yellow"]}, "DEM")
thematic_map_1.addLayer(slope.clip(aoi), {'min': 0, 'max': 10, "palette": ["Red", "Green", "Yellow"]}, "Slope")
thematic_map_1

Distance to Roads

In [ ]:
# Distance to roads
# Data source: https://gee-community-catalog.org/projects/grip/
roads_africa = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/Africa")

# Roads that intersect with study extent
roads_aoi = roads_africa.filterBounds(aoi_bbox)

# Convert road from vector to raster & compute eucledian distance 
roads_raster = ee.Image().float().paint(roads_aoi, 1).clip(aoi)
distance_to_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())  # convert pixel distance to meters
    .rename("dist_to_roads")
    .clip(aoi)
)

# Maximum distance in 'distance_to_roads' layer
print(distance_to_roads.reduceRegion(ee.Reducer.max(), aoi, 1000, maxPixels=1e9).getInfo())

thematic_map_1.addLayer(roads_raster, vis_params_roads_raster, "Road Raster")
thematic_map_1.addLayer(distance_to_roads.select("dist_to_roads"), vis_params_dist_road, "Distance to Road")
thematic_map_1.addLayer(roads_aoi, vis_params_roads_vector, "Road Vector")
thematic_map_1

Distance to Water

In [ ]:
# Permanent water (JRC Global Surface Water)
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(aoi)
permanent_water = gsw.select("occurrence").gte(50)  # >=50% of the time = water

# compute eucledian distance

distance_to_water = (
    permanent_water.Not()
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_to_water")
    .reproject(crs="EPSG:32632", scale=30)   # <-- lock the computation grid here
    .clip(aoi)
)
""" 
distance_to_water = (
    permanent_water.Not()
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_to_water")
    .clip(aoi)
)
"""
# 
print(distance_to_water.reduceRegion(ee.Reducer.max(), aoi, 1000, maxPixels=1e9).getInfo())

thematic_map_1.addLayer(permanent_water, vis_params_water, "Permanent Water")
thematic_map_1.addLayer(distance_to_water.select("dist_to_water"), vis_params_dist_water, "Distance to Water")
thematic_map_1

In [ ]:
date = ee.Date("2015-01-01")
date_plus_1year = date.advance(-1, "year")

print(date.format("YYYY-MM-dd").getInfo())
print(date_plus_1year.format("YYYY-MM-dd").getInfo())

Night Time Light

In [ ]:
# Nighttime lights (VIIRS DNB, annual composite)
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG
def get_nighttime_lights(year, study_extent):
    '''Annual mean VIIRS radiance composite.'''
    start = ee.Date.fromYMD(year, 1, 1) # "2015", "january", "1"
    end = start.advance(1, "year")
    composite = (
        ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
        .filterDate(start, end)
        .select("avg_rad")
        .mean()
        .rename("ntl")
        .clip(study_extent)
    )
    return composite


# Apply function 
ntl_2015 = get_nighttime_lights(2015, aoi)
ntl_2020 = get_nighttime_lights(2020, aoi)
ntl_2022 = get_nighttime_lights(2022, aoi)


thematic_map_2.addLayer(ntl_2015, vis_params_ntl, "NTL - 2015")
thematic_map_2.addLayer(ntl_2020, vis_params_ntl, "NTL - 2020")
thematic_map_2.addLayer(ntl_2022, vis_params_ntl, "NTL - 2022")
thematic_map_2

Population Density

In [ ]:
# Population density (CIESIN GPWv4.11) 
# Data source: https://developers.google.com/earth-engine/datasets/catalog/CIESIN_GPWv411_GPW_Population_Density
# Closest available year to analysis baseline (2015 / 2020 / 2022).
gpw = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Density")

def get_population_density(year, extent):
    '''Return the GPWv4.11 population density image closest to the given year.'''
    available_years = [2000, 2005, 2010, 2015, 2020]
    closest_year = min(available_years, key=lambda y: abs(y - year))
    image = (
        gpw.filter(ee.Filter.calendarRange(closest_year, closest_year, "year"))
        .first()
        .select("population_density")
        .rename("pop_density")
        .clip(extent)
    )
    return image

# Apply function
pop_density_2015 = get_population_density(2015, aoi)
pop_density_2020 = get_population_density(2020, aoi)

thematic_map_2.addLayer(pop_density_2015, vis_params_gpw, "Population Density - 2015")
thematic_map_2.addLayer(pop_density_2020, vis_params_gpw, "Population Density - 2020")
thematic_map_2


Spectral Indices

In [ ]:
# Spectral indices from Landsat (annual cloud-masked median composite) 
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2


def mask_landsat_clouds(image):
    qa = image.select("QA_PIXEL")
    cloud_bit, shadow_bit = 1 << 3, 1 << 4
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(shadow_bit).eq(0))
    return image.updateMask(mask)

def get_spectral_indices(year):
    '''NDVI, NDBI, NDWI annual median composite from Landsat 8/9 SR.'''
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
    landsat = l8.merge(l9).map(mask_landsat_clouds)

    # Apply scale factors
    def apply_scale_factors(image):
        optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
        return image.addBands(optical, None, True)

    landsat = landsat.map(apply_scale_factors)
    composite = landsat.median().clip(aoi)

    ndvi = composite.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")
    ndbi = composite.normalizedDifference(["SR_B6", "SR_B5"]).rename("ndbi")
    ndwi = composite.normalizedDifference(["SR_B3", "SR_B5"]).rename("ndwi")

    return ee.Image.cat([ndvi, ndbi, ndwi])

indices = get_spectral_indices(END_YEAR)


thematic_map_2.addLayer(indices.select("ndvi"), vis_params_ndvi, "NDVI")
thematic_map_2.addLayer(indices.select("ndbi"), vis_params_ndbi, "NDBI")
thematic_map_2.addLayer(indices.select("ndwi"), vis_params_ndwi, "NDWI")

thematic_map_2

Data Preprocessing

In [ ]:
# Combine/stack layers
# 2022 predictor stack 
predictor_stack = (
    elevation
    .addBands(slope)
    .addBands(distance_to_roads)
    .addBands(distance_to_water)
    .addBands(pop_density_2020)
    .addBands(ntl_2022)
    .addBands(indices)
)

#print("Predictor bands:", predictor_stack.bandNames().getInfo())

In [ ]:
# Resample stacked layers/images
predictor_stack_resampled = (predictor_stack.resample("bilinear")
                             .reproject(crs=predictor_stack.projection(), 
                                        scale=30))

In [ ]:
# Fetch Urban layers. Each pixel here is either 0 (non-urban) or 1 (urban)
urban_2015 = urban_by_year[2015]
urban_2022 = urban_by_year[END_YEAR]

# Four Possible transitions
#   0 = Non-urban(2015) - Non-urban(2022)
#   1 = Non-urban(2015) - Urban(2022)
#   2 = Urban(2015)     - Non-urban(2022)
#   3 = Urban(2015)     - Urban(2022)


# Encode all 4 transition combinations in a single raster
transition_15_22 = urban_2015.multiply(2).add(urban_2022).rename("transition")


# Check unique pixel values
transition_hist = transition_15_22.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=10,
    maxPixels=1e13
)

unique_transition_values = ee.Dictionary(transition_hist.get('transition')).keys().getInfo()
#print(unique_transition_values)

In [ ]:
# Keep two transitions: codes 0 and 1
non_urban_2015 = urban_2015.eq(0)

# The transition code (0 or 1) is already the label
change_label = transition_15_22.updateMask(non_urban_2015).rename("label")

#print("Transition codes kept: 0 = stable non-urban, 1 = new urban (2015 - 2022)")

In [ ]:
# Area of 4 transition classes
transition_area = ee.Image.pixelArea().addBands(transition_15_22).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="transition"),
    geometry=aoi,
    scale=30,
    maxPixels=1e10,
)

transition_groups = ee.List(transition_area.get("groups")).getInfo()
transition_names = {
    0: "Non-urban - Non-urban",
    1: "Non-urban - Urban (growth)",
    2: "Urban - Non-urban (loss)",
    3: "Urban - Urban (stable)",
}
for group in transition_groups:
    code_val = int(group["transition"])
    area_km2 = group["sum"] / 1e6
    #print(f"{transition_names[code_val]:32s}: {area_km2:9.2f} km^2")

In [ ]:
# Final training image: predictor stack + land cover transition layer
training_image = (predictor_stack
                  .addBands(change_label)
                  .updateMask(non_urban_2015))

#print(training_image.bandNames().getInfo())

ML Data Preparation
Generate Random Samples

In [ ]:
# Stratified random sample
NUM_POINTS_PER_CLASS  = 1500

samples = training_image.stratifiedSample(
    numPoints = NUM_POINTS_PER_CLASS,
    classBand = "label",
    region = aoi,
    scale = 30,
    seed = 42, 
    geometries = True
)
print("Total training samples:", samples.size().getInfo())
print("Class distribution:", samples.aggregate_histogram("label").getInfo())

In [ ]:
# To pandas DataFrame from FC
def fc_to_df(fc):
    features = fc.getInfo()["features"]
    rows = [f["properties"] for f in features]
    return pd.DataFrame(rows)


samples_df = fc_to_df(samples)
samples_df = samples_df.dropna()
print(samples_df.shape)
#display(samples_df.head())

Multi-collinearity Check

In [ ]:
# Predictors 
#print(samples_df.columns.tolist())
PREDICTOR_VARS = [col for col in samples_df.columns.tolist() if col != "label"]
PREDICTOR_VARS

In [ ]:
# Check Multi-collinearity
X_raw = samples_df[PREDICTOR_VARS].copy()

# Correlation
corr_table = X_raw.corr()
print("Correlation Table")
display(corr_table.round(3)) 

#
plt.figure(figsize=(14, 12))
sns.heatmap(corr_table, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix Heatmap", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Check VIF
# Constant (intercept) for VIF calculation
X_vif = sm.add_constant(X_raw)
vif_df = pd.DataFrame()
vif_df['Variable'] = X_vif.columns
vif_df['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

# Intercept 
vif_df = vif_df[vif_df['Variable'] != 'const'].sort_values('VIF', ascending=False)
print("Variance Inflation Factors (VIF)")
display(vif_df) 

Train/test samples

In [ ]:
# Split samples into train/test. 75% training and 25% testing
samples_split = samples.randomColumn("random", seed=42)
train_samples = samples_split.filter(ee.Filter.lt("random", 0.75))
test_samples = samples_split.filter(ee.Filter.gte("random", 0.75))

print("Train samples:", train_samples.size().getInfo())
print("Test samples :", test_samples.size().getInfo())

In [ ]:
# Visualize the training and testing samples
training_style = {"color": "blue", "pointSize": 3}
testing_style = {"color": "red", "pointSize": 3}

training_layer = train_samples.style(**training_style)
testing_layer = test_samples.style(**testing_style)

ml_map.addLayer(fct_l0.style(color="black", fillColor="00000000"), {}, "FCT Boundary")
ml_map.addLayer(training_layer, {}, "Training Samples")
ml_map.addLayer(testing_layer, {}, "Testing Samples")

ml_map

In [ ]:
training_image.bandNames().getInfo()[:-1]

In [ ]:
#print(training_image.bandNames().getInfo()[:-1])
feature_cols = training_image.bandNames().getInfo()[:-1]
#feature_cols = [col for col in samples_df.columns.tolist() if col != "label"]

Random Forest (Earth Engine)

In [ ]:
# Initialize & Train Random Forest model
ee_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=500,
    minLeafPopulation=5,
    seed=42,
).setOutputMode("PROBABILITY")

ee_classifier_eval = ee_classifier.train(
    features=train_samples,
    classProperty="label",
    inputProperties=feature_cols,
)

print("Random forest trained on the train split.")

In [ ]:
# Classify the held-out test portion
test_classified = test_samples.classify(ee_classifier_eval, outputName="suitability")
test_df = fc_to_df(test_classified).dropna()

y_true = test_df["label"].astype(int)
y_prob = test_df["suitability"]
y_pred = (y_prob >= 0.5).astype(int)

print("Held-out test-split results:")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("ROC-AUC :", roc_auc_score(y_true, y_prob))
print()
print(classification_report(y_true, y_pred))

In [ ]:
# Confusion matrix 
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["No Change", "New Urban"],
            yticklabels=["No Change", "New Urban"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(OUT_THEMATIC/ "confusion_matrix.png", dpi=200)
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color="firebrick", label=f"AUC = {roc_auc_score(y_true, y_prob):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve -- Held-out Test Split (2015->2022)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_THEMATIC/ "roc_curve.png", dpi=200)
plt.show()

In [ ]:
# Retrain on the FULL sample set for the production model
ee_classifier_trained = ee_classifier.train(
    features=samples,
    classProperty="label",
    inputProperties=feature_cols,
)

print("Final random forest retrained on the full 2015 2022 sample set.")

In [ ]:
# Feature importance
importance_dict = ee_classifier_trained.explain().getInfo()["importance"]

importance_df = pd.DataFrame(
    list(importance_dict.items()), columns=["feature", "importance"]
).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x="importance", y="feature", color="firebrick")
plt.title("Feature Importance -- Drivers of Urban Growth (2015->2022)")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.savefig(OUT_THEMATIC/ "feature_importance.png", dpi=200)
plt.show()

importance_df

Suitability Map (Apply trained model to every pixel)

In [ ]:
growth_suitability = predictor_stack.select(feature_cols).classify(
    ee_classifier_trained
).rename("suitability")

In [ ]:
# Remove Existing urban (2022) 
# And see only urban growth model predicted (happened) after 2022
non_urban_2022 = urban_2022.eq(0)
candidate_suitability = growth_suitability.updateMask(non_urban_2022)


suitability_map.addLayer(fct_l0.style(color="black", fillColor="00000000"), {}, "FCT Boundary")
suitability_map.addLayer(candidate_suitability, suitability_vis, "Suitability")
#suitability_map.add_colorbar(suitability_vis, label="Probability of becoming urban")
suitability_map.addLayerControl()
suitability_map

In [ ]:
suitability_map.addLayer(fct_l0.style(color="black", fillColor="00000000"), {}, "FCT Boundary")
suitability_map.addLayer(growth_suitability, suitability_vis, "Urban Growth Suitability")
suitability_map.add_colorbar(suitability_vis, label="Probability of becoming urban")
suitability_map

In [ ]:
dist_water_export = export_image_to_drive(distance_to_water.select("dist_to_water").clip(aoi), 
                                          "Dist_Water_30m_1", "3MTT_C3_Grp1", aoi, 30, "EPSG:32632")
#slope_export = export_image_to_drive(slope.clip(aoi), "Slope_30m", "3MTT_C3_Grp1", aoi, 30, "EPSG:32632")

In [ ]:
# Define your specific target point (Longitude, Latitude)
target_point = ee.Geometry.Point([7.37, 9.19])

# Create a square boundary around the point 
# .bounds() turns the circular buffer radius into a clean square bounding box
local_region = target_point.buffer(10000).bounds()

#  Clip your suitability image to only this local bounding box
clipped_suitability = candidate_suitability.clip(local_region)

# Center your map directly on the point with a closer zoom level 
suitability_map.setCenter(7.37, 9.19, 12)

# Add layers to the map view
# add a point marker to visually see the exact coordinate center
suitability_map.addLayer(target_point, {'color': 'red'}, 'Target Point Center')
suitability_map.addLayer(clipped_suitability, suitability_vis, "Local Suitability")
suitability_map.addLayerControl()



# Display the map in the notebook
suitability_map


#